In [1]:
import os
import time
import pickle
import math
from contextlib import nullcontext
from dataclasses import fields

import torch
from datasets import load_dataset, DatasetDict
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from models.utils import get_lr
from dataloader import pretraining_get_batch, CustomDataset, collate_fn, get_batch, infinite_iterator
from models.memoryGPT.eval import estimate_loss
from models.memoryGPT.gpt2 import GPT
from models.memoryGPT.config import GPTConfig, TrainConfig

In [2]:
# 从配置文件加载配置
config_file = '../configs/finetune_gpt2.py'
config_vars = {}
with open(config_file, 'r', encoding='utf-8') as f:
    exec(f.read(), {}, config_vars)

# 将配置文件中的所有变量加载到config对象中
config_dict = {k: v for k, v in config_vars.items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))}
train_config_fields = {field.name for field in fields(TrainConfig)}
filtered_config_dict = {k: v for k, v in config_dict.items() if k in train_config_fields}
config = TrainConfig(**filtered_config_dict)

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:

'''数据集: 格式, 是否进行了划分
Open-Orca/OpenOrca: ['system_prompt', 'question', 'response'], ['train']
neural-bridge/rag-dataset-12000: ['context', 'question', 'answer'], ['train', 'test']
'''

# 加载数据集
dataset = load_dataset(
    config.data_path,  # Open-Orca/OpenOrca, neural-bridge/rag-dataset-12000
    split="train",
    cache_dir='.cache/huggingface/datasets',
)

# 默认划分数据集 即使有的数据集已经划分了
train_valtest = dataset.train_test_split(test_size=0.2, seed=config.seed)
val_test = train_valtest['test'].train_test_split(test_size=0.5, seed=config.seed)
dataset = DatasetDict({
    'train': train_valtest['train'],
    'val': val_test['train'],
    'test': val_test['test'],
})




In [ ]:
# 创建数据集和DataLoader
train_dataset = CustomDataset(dataset['train'], tokenizer, fields=['context', 'question', 'answer'])
val_dataset = CustomDataset(dataset['val'], tokenizer, fields=['context', 'question', 'answer'])

# train_dataset 按照文本 answer 的长度进行排序
# train_dataset.sort(key='response')
train_dataset.filter_by_length(max_length=1024, keys=['answer',])



train_loader = DataLoader(train_dataset, batch_size=config.batch_size, collate_fn=lambda x: collate_fn(x, tokenizer), shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, collate_fn=lambda x: collate_fn(x, tokenizer), shuffle=True)

train_iter = infinite_iterator(train_loader)
val_iter = infinite_iterator(val_loader)

# train_iter = iter(train_loader)
# val_iter = iter(val_loader)